# 20 · Experiment Tracking, MLflow y AutoML

Cuando un proyecto pasa de 3 experimentos a 300, el problema deja de ser entrenar y pasa a ser **recordar exactamente qué hiciste**. Este lab introduce tracking reproducible y AutoML como automatización controlada, no como caja mágica.

## Objetivos
- Registrar parámetros, métricas, artefactos y modelos.
- Crear una comparación reproducible entre experimentos.
- Introducir MLflow Tracking y Model Registry conceptualmente.
- Usar Optuna para búsqueda.
- Entender qué automatiza AutoML y qué no.
- Diseñar gates para promotion de modelos.


In [ ]:
!pip -q install mlflow optuna
import numpy as np, pandas as pd, os, tempfile, json
import mlflow, mlflow.sklearn, optuna
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
SEED=42
X,y=load_breast_cancer(return_X_y=True,as_frame=True); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
mlflow.set_tracking_uri('file:/tmp/mlruns'); mlflow.set_experiment('alexis-ml-lab')

## 1. Qué registrar
Un run útil incluye:
- commit/git SHA y versión del notebook;
- dataset/version/query/hash;
- random seed;
- hiperparámetros;
- métricas de train/CV/test;
- plots, matriz de confusión, feature importance;
- modelo serializado;
- schema y dependencias;
- notas sobre objetivo y decisión.

Si no puedes reconstruir un resultado meses después, no es realmente reproducible.


In [ ]:
configs=[{'n_estimators':200,'max_depth':4,'min_samples_leaf':2},{'n_estimators':500,'max_depth':None,'min_samples_leaf':2},{'n_estimators':400,'max_depth':8,'min_samples_leaf':5}]
for cfg in configs:
 with mlflow.start_run():
  m=RandomForestClassifier(**cfg,random_state=SEED,n_jobs=-1).fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
  metrics={'roc_auc':roc_auc_score(yte,p),'pr_auc':average_precision_score(yte,p),'log_loss':log_loss(yte,p)}
  mlflow.log_params(cfg); mlflow.log_param('seed',SEED); mlflow.log_metrics(metrics); mlflow.sklearn.log_model(m,'model')
print('runs registrados')

In [ ]:
runs=mlflow.search_runs(experiment_names=['alexis-ml-lab']); display(runs[['run_id','metrics.roc_auc','metrics.pr_auc','metrics.log_loss','params.n_estimators','params.max_depth']].sort_values('metrics.pr_auc',ascending=False))

## 2. Optuna + MLflow
Cada trial puede ser un run; así conservamos historial en vez de solo `best_params`. Para proyectos caros conviene pruning y límites de presupuesto.


In [ ]:
def objective(trial):
 cfg={'n_estimators':trial.suggest_int('n_estimators',150,700),'max_depth':trial.suggest_int('max_depth',3,14),'min_samples_leaf':trial.suggest_int('min_samples_leaf',1,12),'max_features':trial.suggest_categorical('max_features',['sqrt','log2',None])}
 with mlflow.start_run(nested=False):
  m=RandomForestClassifier(**cfg,random_state=SEED,n_jobs=-1).fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; score=average_precision_score(yte,p)
  mlflow.log_params(cfg); mlflow.log_metric('pr_auc_validation_demo',score)
 return score
study=optuna.create_study(direction='maximize'); study.optimize(objective,n_trials=10); print(study.best_value,study.best_params)

> Para un proyecto real, Optuna debe optimizar **validation/CV**, no el test como en este ejemplo compacto. Mantén un test final separado.

## 3. Model Registry
Un registry agrega lifecycle: `candidate/staging/champion/archived`, metadata, approvals y lineage. Una promoción puede exigir:
- métrica principal superior;
- no degradar métricas críticas;
- fairness dentro de límites;
- latencia/costo aceptables;
- tests de schema y seguridad;
- aprobación humana en dominios sensibles.


## 4. ¿Qué es AutoML?
Puede automatizar partes de:
- preprocessing;
- selección/engineering de features;
- elección de algoritmos;
- hyperparameter search;
- ensembles/stacking;
- evaluación y deployment en plataformas gestionadas.

Herramientas: AutoGluon, FLAML, H2O AutoML, auto-sklearn, TPOT, Vertex AI AutoML, Azure AutoML, SageMaker Autopilot.

AutoML **no** decide correctamente por sí solo: target, leakage, split, causalidad, ética, métrica de negocio, calidad de labels o impacto operacional.


In [ ]:
# Ejemplo opcional ligero con FLAML
# !pip -q install flaml[automl]
# from flaml import AutoML
# automl=AutoML(); automl.fit(X_train=Xtr,y_train=ytr,task='classification',metric='roc_auc',time_budget=60)
# print(automl.best_estimator,automl.best_config)

## 5. Reproducibilidad completa
Seeds no garantizan determinismo absoluto en GPU/distribuido. Además guarda:
- lockfile/requirements;
- imagen Docker;
- configuración de hardware;
- versiones CUDA/cuDNN;
- snapshots de datos;
- código de preprocessing;
- definición exacta de métricas.

## 6. CI para ML
Tests recomendados:
- unit tests de transforms;
- data/schema tests;
- notebook smoke test;
- training en muestra pequeña;
- threshold mínimo de performance;
- serialización/deserialización;
- API contract;
- security/dependency scan.

## Ejercicios
1. Loggea matriz de confusión como artefacto.
2. Loggea modelo + signature/input example en MLflow.
3. Integra Optuna usando CV correcta.
4. Compara AutoGluon/FLAML con tu baseline manual.
5. Diseña gates para promover `candidate→champion`.
6. Crea un GitHub Action que ejecute tests del pipeline.
7. Registra dataset hash y Git SHA en cada run.
